# 02 Extract WLASL300 Keypoints

## Purpose

This notebook converts WLASL300 raw `.mp4` sign videos into MediaPipe keypoint arrays. Each video becomes one `.npy` file with the shape `(60, 258)`.

## Why this notebook matters

The model does not train directly on raw videos in this phase. Instead, it trains on structured hand and pose landmarks extracted from each video. This makes training faster, easier to debug, and more suitable for future real-time recognition.

## Extracted features

For every sampled frame, the notebook extracts:

```text
Left hand: 21 landmarks × 3 = 63
Right hand: 21 landmarks × 3 = 63
Pose: 33 landmarks × 4 = 132

Total per frame = 258 features
```

Each video is sampled to 60 frames:

```text
Final video keypoint shape = (60, 258)
```

In [2]:
from pathlib import Path
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

## 1. Set WLASL300 paths

This notebook reads the usable video index from notebook 01 and saves keypoint files into the WLASL300 processed folder.

In [3]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

DATASET_NAME = "WLASL300"
PREFIX = "wlasl300"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
KEYPOINT_DIR = PROCESSED_DIR / "keypoints"

VIDEO_INDEX_FILE = PROCESSED_DIR / f"{PREFIX}_video_index.csv"
KEYPOINT_INDEX_FILE = PROCESSED_DIR / f"{PREFIX}_keypoint_index.csv"
FAILED_LOG_FILE = PROCESSED_DIR / f"{PREFIX}_failed_keypoint_extraction.csv"

KEYPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_NAME)
print("Video index exists:", VIDEO_INDEX_FILE.exists(), VIDEO_INDEX_FILE)
print("Keypoint output folder:", KEYPOINT_DIR)

Dataset: WLASL300
Video index exists: True E:\Be_My_Ear\data\processed\ASL\WLASL300\wlasl300_video_index.csv
Keypoint output folder: E:\Be_My_Ear\data\processed\ASL\WLASL300\keypoints


## 2. Load usable WLASL300 video index

In [4]:
df = pd.read_csv(VIDEO_INDEX_FILE)

print("Videos to process:", len(df))
print("Classes:", df["label_id"].nunique())

df.head()

Videos to process: 2660
Classes: 300


,video_id,original_class_id,gloss,video_path,exists,label_id
0,65096,182,arrive,E:\Be_My_Ear\data\raw\ASL\videos\65096.mp4,True,10
1,65639,279,environment,E:\Be_My_Ear\data\raw\ASL\videos\65639.mp4,True,107
2,10112,191,chat,E:\Be_My_Ear\data\raw\ASL\videos\10112.mp4,True,53
3,55348,247,student,E:\Be_My_Ear\data\raw\ASL\videos\55348.mp4,True,253
4,65092,258,argue,E:\Be_My_Ear\data\raw\ASL\videos\65092.mp4,True,9


## 3. Test video reading with OpenCV

Before running MediaPipe on thousands of frames, we first check whether OpenCV can open a sample video.

In [5]:
test_video = df.iloc[0]["video_path"]

cap = cv2.VideoCapture(test_video)

print("Testing video:")
print(test_video)
print("Opened:", cap.isOpened())
print("Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))

cap.release()

Testing video:
E:\Be_My_Ear\data\raw\ASL\videos\65096.mp4
Opened: True
Frame count: 72
FPS: 23.976023976023978


## 4. Set up MediaPipe Holistic

MediaPipe Holistic detects hands and body pose. We use it because ASL signs are not only about fingers; body and arm movement can also matter.

In [6]:
print("MediaPipe version:", getattr(mp, "__version__", "No version found"))
print("Has mp.solutions:", hasattr(mp, "solutions"))

mp_holistic = mp.solutions.holistic

SEQUENCE_LENGTH = 60

LEFT_HAND_SIZE = 21 * 3
RIGHT_HAND_SIZE = 21 * 3
POSE_SIZE = 33 * 4
FEATURE_SIZE = LEFT_HAND_SIZE + RIGHT_HAND_SIZE + POSE_SIZE

print("Feature size per frame:", FEATURE_SIZE)

MediaPipe version: 0.10.14
Has mp.solutions: True
Feature size per frame: 258


## 5. Landmark extraction functions

Missing landmarks are filled with zeros. This keeps every video in the same shape even when MediaPipe does not detect a hand in some frames.

In [7]:
def extract_landmarks_from_results(results):
    if results.left_hand_landmarks:
        left_hand = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        left_hand = np.zeros(LEFT_HAND_SIZE, dtype=np.float32)

    if results.right_hand_landmarks:
        right_hand = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        right_hand = np.zeros(RIGHT_HAND_SIZE, dtype=np.float32)

    if results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        pose = np.zeros(POSE_SIZE, dtype=np.float32)

    return np.concatenate([left_hand, right_hand, pose])


def extract_keypoints_from_video(video_path, sequence_length=60):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        cap.release()
        return None

    frame_indices = np.linspace(0, frame_count - 1, sequence_length).astype(int)
    sequence = []

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        for frame_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            success, frame = cap.read()

            if not success:
                sequence.append(np.zeros(FEATURE_SIZE, dtype=np.float32))
                continue

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False

            results = holistic.process(image_rgb)
            keypoints = extract_landmarks_from_results(results)
            sequence.append(keypoints)

    cap.release()
    return np.array(sequence, dtype=np.float32)

## 6. Test extraction on one video

Expected output:

```text
(60, 258)
```

In [8]:
sample_row = df.iloc[0]
sample_keypoints = extract_keypoints_from_video(sample_row["video_path"], SEQUENCE_LENGTH)

print("Sample video:", sample_row["video_path"])

if sample_keypoints is None:
    print("Failed to extract sample keypoints.")
else:
    print("Sample keypoint shape:", sample_keypoints.shape)
    print("Data type:", sample_keypoints.dtype)
    print("Min:", sample_keypoints.min())
    print("Max:", sample_keypoints.max())

e:\Be_My_Ear\.venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Sample video: E:\Be_My_Ear\data\raw\ASL\videos\65096.mp4
Sample keypoint shape: (60, 258)
Data type: float32
Min: -1.2544562
Max: 2.2016792


## 7. Test extraction on first 10 videos

Run this before full extraction. If this works, then the MediaPipe pipeline is safe to run on all WLASL300 videos.

In [9]:
test_df = df.head(10).copy()

success_count = 0
fail_count = 0

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Testing first 10 videos"):
    output_path = KEYPOINT_DIR / f"{row['video_id']}.npy"

    keypoints = extract_keypoints_from_video(row["video_path"], SEQUENCE_LENGTH)

    if keypoints is None:
        fail_count += 1
        continue

    np.save(output_path, keypoints)
    success_count += 1

print("Test extraction success:", success_count)
print("Test extraction failed:", fail_count)

Testing first 10 videos: 100%|██████████| 10/10 [00:27<00:00,  2.79s/it]

Test extraction success: 10
Test extraction failed: 0


## 8. Full WLASL300 keypoint extraction

This can take time. The code is resumable: if a `.npy` file already exists, it will skip that video.

In [ ]:
success_count = 0
fail_count = 0
skipped_count = 0
failed_records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting WLASL300 keypoints"):
    video_id = row["video_id"]
    output_path = KEYPOINT_DIR / f"{video_id}.npy"

    if output_path.exists():
        skipped_count += 1
        continue

    keypoints = extract_keypoints_from_video(row["video_path"], SEQUENCE_LENGTH)

    if keypoints is None:
        fail_count += 1
        failed_records.append({
            "video_id": video_id,
            "video_path": row["video_path"],
            "reason": "video_open_or_frame_failed"
        })
        continue

    np.save(output_path, keypoints)
    success_count += 1

print("Extraction completed.")
print("Success:", success_count)
print("Skipped:", skipped_count)
print("Failed:", fail_count)

Extracting WLASL300 keypoints:   1%|▏         | 34/2660 [01:19<1:58:05,  2.70s/it]

## 9. Save extraction logs and keypoint index

The keypoint index connects each `.npy` file with its label and gloss. The training notebook will use this file.

In [ ]:
failed_df = pd.DataFrame(failed_records)
failed_df.to_csv(FAILED_LOG_FILE, index=False)

records = []

for _, row in df.iterrows():
    keypoint_path = KEYPOINT_DIR / f"{row['video_id']}.npy"

    if keypoint_path.exists():
        records.append({
            "video_id": row["video_id"],
            "gloss": row["gloss"],
            "label_id": int(row["label_id"]),
            "original_class_id": int(row["original_class_id"]),
            "video_path": row["video_path"],
            "keypoint_path": str(keypoint_path)
        })

keypoint_df = pd.DataFrame(records)
keypoint_df.to_csv(KEYPOINT_INDEX_FILE, index=False)

print("Saved failure log to:", FAILED_LOG_FILE)
print("Saved keypoint index to:", KEYPOINT_INDEX_FILE)
print("Keypoint samples:", len(keypoint_df))
print("Classes:", keypoint_df["label_id"].nunique())

keypoint_df.head()

## Final summary

After this notebook, continue with:

```text
03_check_wlasl300_keypoints.ipynb
```